In [ ]:
# 02 - Representative-Cell Selection

This notebook applies the block-aware representative-cell selection procedure used in the manuscript.

The workflow expects the following local input files:

- `Processed_Outputv3_1.xlsx`
- `Processed_Outputv3_2.xlsx`
- `Processed_Outputv3_3.xlsx`

For each nominal eight-cell sample block, the procedure checks file-wise homogeneity with respect to temperature, charge cut-off current, and discharge C-rate. If a block is homogeneous, all records belonging to the first sample of that block are retained. If a block is heterogeneous, the data are decomposed into condition-specific sub-blocks, and all records belonging to the earliest sample in each sub-block are retained.

The resulting representative subset is used for statistical analysis, sparse gated correction, and isotonic LUT construction.

In [ ]:
# ============================================================
# Setup and input/output paths
# ============================================================

import os
import importlib
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

DATA_DIR = "."
OUTPUT_DIR = "outputs/02_representative_cell_selection"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INPUT_FILES = [
    "Processed_Outputv3_1.xlsx",
    "Processed_Outputv3_2.xlsx",
    "Processed_Outputv3_3.xlsx",
]

FILE_PATHS = [os.path.join(DATA_DIR, f) for f in INPUT_FILES]

missing_files = [p for p in FILE_PATHS if not os.path.exists(p)]

if missing_files:
    print("Missing required input files:")
    for p in missing_files:
        print(" -", os.path.basename(p))

    if IN_COLAB:
        print("\nPlease upload the missing files.")
        uploaded = files.upload()

        missing_after = [p for p in FILE_PATHS if not os.path.exists(p)]
        if missing_after:
            raise FileNotFoundError(
                "The following files are still missing: "
                + ", ".join(os.path.basename(p) for p in missing_after)
            )
    else:
        raise FileNotFoundError(
            "Required input files are missing. Place the three Processed_Outputv3 files "
            "in the notebook working directory."
        )
else:
    print("All required input files are available.")

print("\nFiles used:")
for p in FILE_PATHS:
    print(" -", os.path.basename(p))

In [ ]:
# ============================================================
# Helper functions
# ============================================================

NEEDED_COLS = [
    "sample_number",
    "Rate",
    "Temperature",
    "Charge_Rate",
    "Step_Index",
    "Cycle_Index",
    "VoltageV",
    "CurrentA",
    "SoC",
]

def ensure_package(pkg, import_name=None):
    """Install a package only when it is missing. Mainly useful in Colab."""
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        return True
    except Exception:
        if not IN_COLAB:
            raise ImportError(
                f"Package '{pkg}' is missing. Install it before running this notebook."
            )
        print(f"Installing missing package: {pkg}")
        import sys
        !{sys.executable} -m pip -q install {pkg}
        importlib.import_module(import_name)
        return True


def to_int_sample(x):
    """Convert sample identifiers such as '001.0' or '1' to integer sample IDs."""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    s = s.split(".")[0]
    s = s.lstrip("0")
    return int(s) if s else 0


def rep_of_block(sample_int):
    """
    Return the nominal block representative ID.

    Examples:
    1--8   -> 1
    9--16  -> 9
    17--24 -> 17
    """
    s = int(sample_int)
    return ((s - 1) // 8) * 8 + 1


def read_any_path(path):
    """
    Read one processed input file and retain only the columns required
    by the representative-cell selection workflow.
    """
    name = os.path.basename(path)
    lower = name.lower()

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    if lower.endswith(".csv"):
        df = pd.read_csv(path)

    elif lower.endswith(".xlsx"):
        ensure_package("openpyxl")
        try:
            df = pd.read_excel(path, sheet_name="Step5_Data", engine="openpyxl")
        except Exception:
            df = pd.read_excel(path, engine="openpyxl")

    elif lower.endswith(".xls"):
        ensure_package("xlrd==1.2.0", "xlrd")
        try:
            df = pd.read_excel(path, sheet_name="Step5_Data", engine="xlrd")
        except Exception:
            df = pd.read_excel(path, engine="xlrd")

    else:
        raise ValueError(f"Unsupported file extension: {name}")

    keep = [c for c in NEEDED_COLS if c in df.columns]
    df = df[keep].copy()
    df["__source_file__"] = name

    return df

In [ ]:
# ============================================================
# Condition lookup table
# ============================================================
# Important column convention in the processed files:
#
#   Rate        -> charge cut-off current level: C/5 or C/40
#   Charge_Rate -> discharge C-rate: 0.7C, 1C, or 2C
#
# The Condition_ID order follows the manuscript condition map:
# Temperature: 10, 25, 45, 60 °C
# Discharge C-rate: 0.7C, 1C, 2C
# Charge cut-off current: C/5, C/40

def build_condition_lookup():
    order_T = [10, 25, 45, 60]
    order_DR = ["0.7C", "1C", "2C"]
    order_R = ["C/5", "C/40"]

    rows = []
    cid = 1

    for T in order_T:
        for DR in order_DR:
            for R in order_R:
                rows.append((cid, T, R, DR))
                cid += 1

    return pd.DataFrame(
        rows,
        columns=["Condition_ID", "Temperature", "Rate", "Charge_Rate"]
    )

COND_LUT = build_condition_lookup()

print("Condition lookup table:")
display(COND_LUT)

In [ ]:
# ============================================================
# Read and combine input files
# ============================================================

dfs = []

for p in FILE_PATHS:
    print(f"Reading: {os.path.basename(p)}")
    dfp = read_any_path(p)
    dfs.append(dfp)

raw = pd.concat(dfs, ignore_index=True)

required_min = {"sample_number", "Rate", "Temperature", "Charge_Rate"}
missing = sorted(required_min - set(raw.columns))

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nCombined raw data:")
print(f"Rows: {len(raw):,}")
print(f"Columns: {list(raw.columns)}")
print(f"Unique sample_number values: {raw['sample_number'].nunique()}")

In [ ]:
# ============================================================
# Sample and nominal block preprocessing
# ============================================================

raw["sample_int"] = raw["sample_number"].map(to_int_sample)
raw["rep_sample"] = raw["sample_int"].map(rep_of_block)

print("Sample/block preprocessing completed.")
print(f"Unique integer samples: {raw['sample_int'].nunique()}")
print(f"Nominal block representatives: {raw['rep_sample'].nunique()}")

display(
    raw[["__source_file__", "sample_number", "sample_int", "rep_sample",
         "Temperature", "Rate", "Charge_Rate"]]
    .drop_duplicates()
    .head(20)
)

In [ ]:
# ============================================================
# File-wise eight-cell block homogeneity check
# ============================================================

uniq_by_sample = raw.drop_duplicates(
    subset=["__source_file__", "sample_int", "Temperature", "Rate", "Charge_Rate"]
)

hom_file = (
    uniq_by_sample
    .groupby(["__source_file__", "rep_sample"])[["Temperature", "Rate", "Charge_Rate"]]
    .nunique()
    .rename(
        columns={
            "Temperature": "n_T",
            "Rate": "n_Rate",
            "Charge_Rate": "n_Discharge",
        }
    )
    .reset_index()
    .sort_values(["__source_file__", "rep_sample"])
)

hom_file["homogeneous_block"] = (
    hom_file["n_T"].eq(1)
    & hom_file["n_Rate"].eq(1)
    & hom_file["n_Discharge"].eq(1)
)

print("File-wise block homogeneity summary:")
display(hom_file.head(25))

print("\nHomogeneous block counts:")
display(
    hom_file.groupby(["__source_file__", "homogeneous_block"])
    .size()
    .reset_index(name="n_blocks")
)

In [ ]:
# ============================================================
# Representative-cell selection
# ============================================================
# Rule A:
#   Homogeneous block -> retain all records of the first sample in the nominal block.
#
# Rule B:
#   Heterogeneous block -> split by (Temperature, Rate, Charge_Rate),
#   then retain all records of the earliest sample in each sub-block.

# A) Homogeneous blocks
homo_keys = set(
    map(
        tuple,
        hom_file.loc[
            hom_file["homogeneous_block"],
            ["__source_file__", "rep_sample"]
        ].values
    )
)

is_homo = raw.apply(
    lambda r: (r["__source_file__"], r["rep_sample"]) in homo_keys,
    axis=1
)

rep_homo = raw[
    is_homo & (raw["sample_int"] == raw["rep_sample"])
].copy()

# B) Heterogeneous blocks
hetero = hom_file.loc[
    ~hom_file["homogeneous_block"],
    ["__source_file__", "rep_sample"]
]

if not hetero.empty:
    hetero_keys = set(map(tuple, hetero.values))

    is_hetero = raw.apply(
        lambda r: (r["__source_file__"], r["rep_sample"]) in hetero_keys,
        axis=1
    )

    cand = raw[is_hetero].copy()

    grp_cols = [
        "__source_file__",
        "rep_sample",
        "Temperature",
        "Rate",
        "Charge_Rate",
    ]

    first_map = (
        cand
        .dropna(subset=["sample_int"])
        .groupby(grp_cols)["sample_int"]
        .min()
        .rename("first_sample_in_subblock")
    )

    cand = cand.merge(first_map, on=grp_cols, how="left")

    rep_hetero = cand[
        cand["sample_int"] == cand["first_sample_in_subblock"]
    ].copy()

else:
    rep_hetero = raw.iloc[0:0].copy()

rep_df = pd.concat([rep_homo, rep_hetero], ignore_index=True)

print("Representative-cell selection completed.")
print(f"Retained records: {len(rep_df):,}")
print(f"Retained unique representative sample IDs: {rep_df['sample_int'].nunique()}")

display(rep_df.head(10))

In [ ]:
# ============================================================
# Assign Condition_ID and compute condition coverage
# ============================================================

rep_df = rep_df.merge(
    COND_LUT,
    on=["Temperature", "Rate", "Charge_Rate"],
    how="left"
)

if rep_df["Condition_ID"].isna().any():
    unmatched = (
        rep_df.loc[
            rep_df["Condition_ID"].isna(),
            ["Temperature", "Rate", "Charge_Rate"]
        ]
        .drop_duplicates()
        .sort_values(["Temperature", "Rate", "Charge_Rate"])
    )
    print("Unmatched condition combinations:")
    display(unmatched)
    raise ValueError("Some retained records could not be mapped to Condition_ID.")

coverage_all = (
    rep_df
    .groupby(["Condition_ID", "Temperature", "Rate", "Charge_Rate"])
    .agg(
        n_rows=("sample_int", "size"),
        n_unique_samples=("sample_int", "nunique")
    )
    .reset_index()
    .sort_values("Condition_ID")
)

coverage_by_file = (
    rep_df
    .groupby(["__source_file__", "Condition_ID", "Temperature", "Rate", "Charge_Rate"])
    .agg(
        n_rows=("sample_int", "size"),
        n_unique_samples=("sample_int", "nunique")
    )
    .reset_index()
    .sort_values(["__source_file__", "Condition_ID"])
)

print("Combined condition coverage:")
display(coverage_all)

print("File-wise condition coverage:")
display(coverage_by_file.head(50))

In [ ]:
# ============================================================
# Final representative subset summary
# ============================================================

n_retained_samples = rep_df["sample_int"].nunique()
n_retained_records = len(rep_df)
n_covered_conditions = rep_df["Condition_ID"].dropna().nunique()

condition_coverage = (
    rep_df
    .dropna(subset=["Condition_ID"])
    .groupby("Condition_ID")
    .agg(
        n_representative_samples=("sample_int", "nunique"),
        n_records=("sample_int", "size"),
        representative_samples=(
            "sample_int",
            lambda x: ", ".join(map(str, sorted(pd.unique(x))))
        )
    )
    .reset_index()
    .sort_values("Condition_ID")
)

condition_coverage["Condition_ID"] = condition_coverage["Condition_ID"].astype(int)

expected_conditions = set(range(1, 25))
observed_conditions = set(condition_coverage["Condition_ID"].astype(int))
missing_conditions = sorted(expected_conditions - observed_conditions)

print("=== Final Representative Modeling Subset Summary ===")
print(f"Retained representative samples : {n_retained_samples}")
print(f"Retained records                : {n_retained_records:,}")
print(f"Covered conditions              : {n_covered_conditions} / 24")

print("\n=== Condition-Level Representative Coverage ===")
display(condition_coverage)

print("\n=== Missing Conditions ===")
if missing_conditions:
    print("Missing Condition_ID:", missing_conditions)
else:
    print("No missing conditions. All 24 nominal conditions are covered.")

manuscript_sentence = (
    f"After representative-cell selection, the final analysis subset contained "
    f"{n_retained_samples} retained representative samples, "
    f"{n_retained_records:,} retained records, and covered "
    f"{n_covered_conditions} of the 24 nominal operating conditions."
)

print("\n=== Manuscript Sentence ===")
print(manuscript_sentence)

In [ ]:
# ============================================================
# Save outputs
# ============================================================

rep_csv = os.path.join(OUTPUT_DIR, "rep_filtered_byfile_aware.csv")
hom_csv = os.path.join(OUTPUT_DIR, "block_homogeneity_by_file.csv")
cov_all_csv = os.path.join(OUTPUT_DIR, "condition_coverage_all.csv")
cov_byfile_csv = os.path.join(OUTPUT_DIR, "condition_coverage_by_file.csv")
summary_txt = os.path.join(OUTPUT_DIR, "representative_subset_summary.txt")
condition_coverage_csv = os.path.join(OUTPUT_DIR, "representative_condition_coverage.csv")

rep_df.to_csv(rep_csv, index=False)
hom_file.to_csv(hom_csv, index=False)
coverage_all.to_csv(cov_all_csv, index=False)
coverage_by_file.to_csv(cov_byfile_csv, index=False)
condition_coverage.to_csv(condition_coverage_csv, index=False)

rep_parq = None
try:
    rep_parq = os.path.join(OUTPUT_DIR, "rep_filtered_byfile_aware.parquet")
    rep_df.to_parquet(rep_parq, index=False)
except Exception:
    rep_parq = None

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write("Final Representative Modeling Subset Summary\n")
    f.write(f"Retained representative samples: {n_retained_samples}\n")
    f.write(f"Retained records: {n_retained_records}\n")
    f.write(f"Covered conditions: {n_covered_conditions} / 24\n")
    f.write(f"Missing conditions: {missing_conditions}\n\n")
    f.write("Manuscript sentence:\n")
    f.write(manuscript_sentence + "\n")

print("Saved files:")
print(" -", rep_csv)
if rep_parq:
    print(" -", rep_parq)
print(" -", hom_csv)
print(" -", cov_all_csv)
print(" -", cov_byfile_csv)
print(" -", summary_txt)
print(" -", condition_coverage_csv)

# Optional Colab download
if IN_COLAB and files is not None:
    for out_path in [
        rep_csv,
        rep_parq,
        hom_csv,
        cov_all_csv,
        cov_byfile_csv,
        summary_txt,
        condition_coverage_csv,
    ]:
        if out_path and os.path.exists(out_path):
            try:
                files.download(out_path)
            except Exception:
                pass

In [ ]:
## Expected manuscript-level checks

This notebook is expected to confirm the representative subset used downstream in the manuscript:

- retained representative sample identifiers: 25
- retained records: 309,020
- covered nominal operating conditions: 24 / 24

The main output file is:

`outputs/02_representative_cell_selection/rep_filtered_byfile_aware.csv`

This file is the representative-cell subset used for the statistical analysis, sparse gated correction experiment, and isotonic LUT construction.

In [ ]:
# ============================================================
# Representative subset summary and condition coverage
# GitHub/reproducibility-ready optional verification cell
# ============================================================

import os
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# Input/output paths
# ------------------------------------------------------------

INPUT_CANDIDATES = [
    "outputs/02_representative_cell_selection/rep_filtered_byfile_aware.csv",
    "outputs/rep_filtered_byfile_aware.csv",
    "rep_filtered_byfile_aware.csv",
]

OUTPUT_DIR = "outputs/02_representative_cell_selection"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load representative subset if rep_df is not already in memory
# ------------------------------------------------------------

if "rep_df" not in globals():
    found_path = None

    for path in INPUT_CANDIDATES:
        if os.path.exists(path):
            found_path = path
            break

    if found_path is None:
        raise FileNotFoundError(
            "Representative subset file was not found. "
            "Run the representative-cell selection notebook first. "
            "Expected one of:\n"
            + "\n".join(INPUT_CANDIDATES)
        )

    rep_df = pd.read_csv(found_path)
    print(f"Representative subset loaded from: {found_path}")

else:
    print("Representative subset is already available in memory as rep_df.")

# ------------------------------------------------------------
# Required column check
# ------------------------------------------------------------

required_cols = {"sample_int", "Condition_ID"}
missing_cols = required_cols - set(rep_df.columns)

if missing_cols:
    raise ValueError(
        f"Missing required columns in rep_df: {missing_cols}"
    )

# ------------------------------------------------------------
# Basic subset summary
# ------------------------------------------------------------

n_retained_samples = rep_df["sample_int"].nunique()
n_retained_records = len(rep_df)
n_covered_conditions = rep_df["Condition_ID"].dropna().nunique()

print("\n=== Final Representative Modeling Subset Summary ===")
print(f"Retained representative sample identifiers : {n_retained_samples}")
print(f"Retained records                           : {n_retained_records:,}")
print(f"Covered nominal operating conditions       : {n_covered_conditions} / 24")

# ------------------------------------------------------------
# Condition-level representative coverage
# ------------------------------------------------------------

condition_coverage = (
    rep_df
    .dropna(subset=["Condition_ID"])
    .groupby("Condition_ID")
    .agg(
        n_representative_samples=("sample_int", "nunique"),
        n_records=("sample_int", "size"),
        representative_samples=(
            "sample_int",
            lambda x: ", ".join(map(str, sorted(pd.unique(x))))
        )
    )
    .reset_index()
    .sort_values("Condition_ID")
)

condition_coverage["Condition_ID"] = condition_coverage["Condition_ID"].astype(int)

print("\n=== Condition-Level Representative Coverage ===")
display(condition_coverage)

# ------------------------------------------------------------
# Missing-condition check
# ------------------------------------------------------------

expected_conditions = set(range(1, 25))
observed_conditions = set(condition_coverage["Condition_ID"].astype(int))
missing_conditions = sorted(expected_conditions - observed_conditions)

print("\n=== Missing Conditions ===")

if missing_conditions:
    print("Missing Condition_ID:", missing_conditions)
else:
    print("No missing conditions. All 24 nominal operating conditions are covered.")

# ------------------------------------------------------------
# Manuscript-ready sentence
# ------------------------------------------------------------

manuscript_sentence = (
    f"After representative-cell selection, the final analysis subset contained "
    f"{n_retained_samples} retained representative sample identifiers, "
    f"{n_retained_records:,} retained records, and covered "
    f"{n_covered_conditions} of the 24 nominal operating conditions."
)

print("\n=== Manuscript Sentence ===")
print(manuscript_sentence)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

summary_path = os.path.join(
    OUTPUT_DIR,
    "representative_subset_summary.txt"
)

coverage_path = os.path.join(
    OUTPUT_DIR,
    "representative_condition_coverage.csv"
)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Final Representative Modeling Subset Summary\n")
    f.write(f"Retained representative sample identifiers: {n_retained_samples}\n")
    f.write(f"Retained records: {n_retained_records}\n")
    f.write(f"Covered nominal operating conditions: {n_covered_conditions} / 24\n")
    f.write(f"Missing conditions: {missing_conditions}\n\n")
    f.write("Manuscript sentence:\n")
    f.write(manuscript_sentence + "\n")

condition_coverage.to_csv(coverage_path, index=False)

print("\nSaved files:")
print(summary_path)
print(coverage_path)